# License Plate Segmentor — YOLOv8n-seg

Trains a segmentation model on the [License Plate Instance Segmentation dataset](https://universe.roboflow.com/car-license-plate-detection/license-plate-lprsn) (~2400 images).

Input size is 416 instead of the default 640 — still a multiple of 32 (required by YOLO's stride), noticeably faster, and we crop OCR regions from the original full-res image anyway so no quality is lost there.

**Before running:** Runtime → Change runtime type → T4 GPU  
**API key:** Colab sidebar → key icon → add secret `ROBOFLOW_API_KEY`

## References

- Redmon et al., *You Only Look Once: Unified, Real-Time Object Detection*, CVPR 2016
- Redmon & Farhadi, *YOLOv3: An Incremental Improvement*, arXiv:1804.02767 — multiscale inputs (320/416/608, all multiples of 32)
- Bochkovskiy et al., *YOLOv4: Optimal Speed and Accuracy of Object Detection*, arXiv:2004.10934
- Jocher et al., *Ultralytics YOLOv8*, 2023, https://github.com/ultralytics/ultralytics
- He et al., *Mask R-CNN*, ICCV 2017 — instance segmentation foundation

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace("car-license-plate-detection").project("license-plate-lprsn").version(18).download("yolov8")

yaml_path = dataset.location + "/data.yaml"

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,
    project="/content/runs",
    name="plate_segmentor",
    patience=10,
    exist_ok=True,
)

In [ ]:
best_weights = "/content/runs/plate_segmentor/weights/best.pt"
trained = YOLO(best_weights)
metrics = trained.val(data=yaml_path)

print(f"Box  mAP50:    {metrics.box.map50:.3f}")
print(f"Box  mAP50-95: {metrics.box.map:.3f}")
print(f"Mask mAP50:    {metrics.seg.map50:.3f}")
print(f"Mask mAP50-95: {metrics.seg.map:.3f}")

In [ ]:
trained.export(format="onnx", imgsz=416, simplify=True, opset=12)

onnx_path = best_weights.replace(".pt", ".onnx")
print("ONNX saved to:", onnx_path)

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
shutil.copy(onnx_path, "/content/drive/MyDrive/plate-segmentor.onnx")
print("Done. Copy to: frontend/public/models/plate-segmentor.onnx")